# Tema 1 — Modelos de Machine Learning con scikit-learn

| Recurso | Fichero |
| --- | --- |
| Ejercicios ML | [`7_ejercicios_modelos.md`](7_ejercicios_modelos.md) |
| Ejercicios datos + ML | [`7_ejercicios_datos_y_modelos.md`](7_ejercicios_datos_y_modelos.md) |
| Dataset local (clustering) | [`Datos/iris.csv`](Datos/iris.csv) |
| Documentación | [scikit-learn.org](https://scikit-learn.org/stable/) |

Este notebook es la **referencia amplia** de modelos en DAVD: clasificación, regresión y clustering con demos ejecutables, métricas, anti-patrones y puente a visualización / Dash. Úsalo en clase y como manual al preparar el modelo del proyecto.

**Cómo trabajarlo:** ejecuta las celdas en orden (`scikit-learn`, `pandas`, `matplotlib` instalados).


---

## 0. Objetivos de aprendizaje

Al terminar esta sesión deberías ser capaz de:

1. Distinguir aprendizaje **supervisado** (clasificación / regresión) y **no supervisado** (clustering).
2. Pasar un dataset de sklearn a `DataFrame` y diagnosticarlo.
3. Hacer un `train_test_split` reproducible (`random_state`).
4. Entrenar un clasificador, un regressor y un `KMeans`.
5. Evaluar con métricas adecuadas (report, RMSE/MAE/MAPE/R², silhouette, codo).
6. Explicar por qué el modelo del proyecto debe ser **reentrenable** y usable desde una app.
7. Evitar fugas de datos y métricas engañosas.

---


## 1. Por qué esta sesión importa en DAVD

### 1.1 Qué entregamos

En DAVD el modelo no es el final: alimenta **decisiones visuales** (KPIs, predicciones en un dashboard).

```text
datos → limpieza (pandas) → modelo (sklearn) → predicción/score → visualización / Dash → despliegue
```

### 1.2 Mapa mental

```text
5_lectura_de_datos     →  cargar y preparar tablas
6_modelos (hoy)        →  entrenar y evaluar
8_visualizaciones      →  comunicar resultados
Tema 2 (Plotly/Dash)   →  interactuar con el modelo
```

### 1.3 Analogías

| Concepto | Analogía |
| --- | --- |
| Features `X` | Ingredientes medibles |
| Target `y` | Etiqueta o valor a predecir |
| `train_test_split` | Ensayo vs examen |
| Overfitting | Memorizar el ensayo y suspender el examen |
| Métrica | Cómo decides si el modelo “vale” |
| Clustering | Agrupar sin etiquetas previas |

### 1.4 Regla de oro

> **Separa train/test, reporta métricas de test, y deja el entrenamiento reproducible (`random_state` + datos versionados).**

---


## 2. Entorno e imports


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn.datasets
from sklearn.cluster import KMeans
from sklearn.linear_model import ElasticNet
from sklearn.metrics import (
    classification_report,
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

DATA_DIR = Path("Datos")
print("sklearn datasets OK | Datos existe:", DATA_DIR.exists())


---

## 3. Aprendizaje supervisado

Hay **features** (`X`) y un **target** (`y`).

| Tipo | Target | Ejemplo DAVD |
| --- | --- | --- |
| Clasificación | Categórico | Tipo de cliente, riesgo alto/bajo |
| Regresión | Continuo | Demanda, precio, score |

---

## 4. Clasificación — Wine Recognition

Documentación del dataset: [`load_wine`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html).

### 4.1 Carga y DataFrame


In [ ]:
wine = sklearn.datasets.load_wine()
df_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
df_wine["target"] = wine.target
df_wine["target_name"] = df_wine["target"].map(dict(enumerate(wine.target_names)))
df_wine.head()


### 4.2 Diagnóstico rápido


In [ ]:
print("shape:", df_wine.shape)
print("nulos:", int(df_wine.isna().sum().sum()))
print("balance de clases:")
print(df_wine["target_name"].value_counts(normalize=True).round(3))


### 4.3 Train / test y entrenamiento

Modelo: [`LinearSVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html).


In [ ]:
X = wine.data
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=123, stratify=y
)

clf = LinearSVC(C=0.1, max_iter=5000, dual=False)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=wine.target_names))


**Lectura del report:** precision / recall / f1 por clase + accuracy global. En el proyecto, elige métricas alineadas al negocio (a veces el recall de una clase importa más que el accuracy).

---

## 5. Regresión — California Housing

> Boston Housing se eliminó de scikit-learn por consideraciones éticas. Usamos **California Housing**.

Documentación: [`fetch_california_housing`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html).

### 5.1 Carga y diagnóstico


In [ ]:
housing = sklearn.datasets.fetch_california_housing()
df_h = pd.DataFrame(housing.data, columns=housing.feature_names)
df_h["target"] = housing.target
print(df_h.shape)
print(df_h.isna().sum().sum(), "nulos")
df_h[["MedInc", "AveRooms", "target"]].describe()


### 5.2 Preguntas de exploración (hazlas por escrito)

- ¿Qué rango tiene el `target` (valor medio de vivienda)?
- ¿Hay features con escalas muy distintas? ¿Haría falta escalar?
- ¿Qué harías con outliers extremos?

### 5.3 Entrenamiento — ElasticNet

Documentación: [`ElasticNet`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html).


In [ ]:
X = housing.data
y = housing.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=123
)

reg = ElasticNet(random_state=123)
reg.fit(X_train, y_train)
pred_train = reg.predict(X_train)
pred_test = reg.predict(X_test)


def regression_report(y_true, y_hat, label: str) -> None:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_hat)))
    mae = float(mean_absolute_error(y_true, y_hat))
    mape = float(mean_absolute_percentage_error(y_true, y_hat))
    r2 = float(r2_score(y_true, y_hat))
    print(f"[{label}] RMSE={rmse:.4f} | MAE={mae:.4f} | MAPE={100*mape:.2f}% | R²={r2:.4f}")


regression_report(y_train, pred_train, "train")
regression_report(y_test, pred_test, "test")


Si train va mucho mejor que test → sospecha de **overfitting** (o de fuga de información).

---

## 6. Aprendizaje no supervisado — clustering

Sin etiquetas de entrenamiento “oficiales”: buscamos estructura (segmentos, anomalías, temas).

### 6.1 Iris local

Ruta portable: `Datos/iris.csv`.


In [ ]:
iris = pd.read_csv(DATA_DIR / "iris.csv")
iris.columns = [c.replace(".", "_") for c in iris.columns]
X = iris.drop(columns=["variety"]).to_numpy()
y_true = iris["variety"].map({"Setosa": 0, "Versicolor": 1, "Virginica": 2}).to_numpy()
iris.head()


### 6.2 K-Means

En clustering puro **no hace falta** `train_test_split` para aprender grupos; a veces se reserva un holdout solo para evaluar estabilidad. Aquí lo usamos de forma pedagógica.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_true, test_size=0.3, random_state=123, stratify=y_true
)

kmeans = KMeans(n_clusters=3, random_state=123, n_init=10)
kmeans.fit(X_train)
pred_train = kmeans.predict(X_train)
pred_test = kmeans.predict(X_test)
pred_train[:20]


### 6.3 Método del codo


In [ ]:
K = range(1, 10)
distortions = []
for k in K:
    model = KMeans(n_clusters=k, random_state=123, n_init=10)
    model.fit(X_train)
    distortions.append(model.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(list(K), distortions, "bx-")
plt.xlabel("k")
plt.ylabel("Inercia (distortion)")
plt.title("Elbow method — búsqueda de k")
plt.grid(True)
plt.show()


### 6.4 Silhouette score

Documentación: [`silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).


In [ ]:
print("silhouette train:", round(silhouette_score(X_train, pred_train), 4))
print("silhouette test :", round(silhouette_score(X_test, pred_test), 4))


---

## 7. Mini-patrón reutilizable para el proyecto

```text
cargar datos → DataFrame → X/y → split → fit → métricas test → guardar modelo (joblib) → servir en Dash
```

Ejemplo mínimo de empaquetado mental:

```python
# train.py  → ajusta y guarda model.joblib
# app.py    → carga model.joblib y predice en un callback
```

---

## 8. Anti-patrones frecuentes

1. Evaluar solo en train.
2. No fijar `random_state` (resultados irreproducibles).
3. Filtrar/escalar con estadísticas del **test** metidas en el train.
4. Accuracy con clases muy desbalanceadas sin mirar el report.
5. Confundir clustering con clasificación (“el cluster = la etiqueta verdadera” siempre).
6. Rutas absolutas a CSV.
7. Entrenar en un notebook y no poder reejecutar el mismo pipeline.
8. Meter el `fit` dentro del callback de Dash en cada click (lento y peligroso).

---

## 9. Autoevaluación

1. ¿Clasificación o regresión para predecir un precio?
2. ¿Para qué sirve el split train/test?
3. ¿Qué métricas usarías en clasificación multiclasse?
4. ¿Qué indica un R² de test mucho peor que el de train?
5. ¿El codo garantiza el k óptimo?
6. ¿Dónde vivirían `train` y `predict` en tu repo del proyecto?

---

## 10. Checklist de salida

- [ ] Wine: split + `LinearSVC` + `classification_report`
- [ ] California Housing: `ElasticNet` + RMSE/MAE/MAPE/R² train y test
- [ ] Iris: `KMeans` + codo + silhouette
- [ ] Rutas con `Path("Datos/...")`
- [ ] Sé explicar supervised vs unsupervised en una frase

---

## 11. Para la siguiente sesión

1. Haz los ejercicios de [`7_ejercicios_modelos.md`](7_ejercicios_modelos.md) (matriz de confusión, PCA, más clusterers).
2. Ojea `8_visualizaciones_sencillas.ipynb` para comunicar métricas y predicciones.
3. Decide qué target tendrá el modelo de **tu** dashboard.

---

## 12. Apéndice A — Chuleta

```python
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
rmse = mean_squared_error(y_test, y_pred) ** 0.5
```

---

## 13. Apéndice B — Glosario

| EN | ES / nota |
| --- | --- |
| supervised learning | aprendizaje supervisado |
| unsupervised learning | aprendizaje no supervisado |
| feature | variable explicativa |
| target / label | objetivo / etiqueta |
| train/test split | partición entrenamiento/prueba |
| overfitting | sobreajuste |
| silhouette score | cohesión/separación de clusters |

---

## 14. Apéndice C — Preguntas típicas

**¿Obligatorio SVM / ElasticNet?**  
No: son demos. En el proyecto elige un modelo justificado.

**¿Hay que desplegar el modelo ya?**  
No en esta sesión; sí debes dejarlo **reentrenable** y evaluado.

**¿Puedo usar solo accuracy?**  
Mejor el report completo (y la matriz de confusión en los ejercicios).

---

## 15. Cierre

> **Datos limpios + split honesto + métricas de test + modelo reproducible = base sólida para el dashboard.**

Siguiente paso: [`7_ejercicios_modelos.md`](7_ejercicios_modelos.md).
